# Airline Operational Performance Analysis (2015)

## Capstone Project
Operational Bottleneck Identification and Performance Optimization in the U.S. Airline Industry

### Objective
This notebook performs descriptive analytics to identify:
- Airline-level delay and cancellation patterns
- Airport congestion effects
- Seasonal and hourly delay trends
- Route-level operational bottlenecks

In [0]:
%sql
USE CATALOG tables;
USE SCHEMA default;

## Executive Snapshot

This section provides an overall industry-level performance summary.

In [0]:
%sql
SELECT 
    COUNT(*) AS total_flights,
    ROUND(SUM(CASE WHEN ARRIVAL_DELAY > 0 THEN 1 ELSE 0 END) / COUNT(*),4) AS overall_delay_rate,
    ROUND(AVG(CANCELLED),4) AS overall_cancel_rate,
    ROUND(AVG(ARRIVAL_DELAY),2) AS avg_arrival_delay_minutes
FROM tables.default.flights_cleaned;


total_flights,overall_delay_rate,overall_cancel_rate,avg_arrival_delay_minutes
5818795,0.3586,0.0154,4.27


### Interpretation

- This provides a benchmark for airline performance comparison.
- Overall delay rate establishes industry-wide operational baseline.
- Cancellation rate indicates systemic reliability level.

## 1. Airline-Level Performance Analysis

This section evaluates operational efficiency across airlines.

In [0]:
%sql
SELECT 
    AIRLINE,
    COUNT(*) AS total_flights,
    ROUND(SUM(CASE WHEN ARRIVAL_DELAY > 0 THEN 1 ELSE 0 END)/COUNT(*),4) AS arrival_delay_rate,
    ROUND(SUM(CASE WHEN DEPARTURE_DELAY > 0 THEN 1 ELSE 0 END)/COUNT(*),4) AS departure_delay_rate,
    ROUND(AVG(CANCELLED),4) AS cancel_rate,
    ROUND(AVG(ARRIVAL_DELAY),2) AS avg_arrival_delay
FROM tables.default.flights_cleaned
GROUP BY AIRLINE
ORDER BY arrival_delay_rate DESC;


AIRLINE,total_flights,arrival_delay_rate,departure_delay_rate,cancel_rate,avg_arrival_delay
NK,117379,0.4846,0.4438,0.0171,14.2
F9,90834,0.4539,0.3841,0.0065,12.38
HA,76264,0.3956,0.2641,0.0022,1.89
VX,61903,0.3906,0.3777,0.0086,4.69
US,198715,0.3839,0.3148,0.0205,3.62
B6,267047,0.3819,0.3822,0.016,6.55
OO,588344,0.3781,0.2916,0.0169,5.71
WN,1261855,0.3731,0.4492,0.0127,4.31
EV,571969,0.3728,0.297,0.0266,6.37
UA,515707,0.3611,0.4974,0.0127,5.31


### Observations

- Airlines show significant variation in delay rates.
- Some carriers demonstrate consistently higher cancellation rates.
- Differences suggest operational strategy or hub exposure impact.

## 2. Airport Congestion Analysis

This section identifies airports contributing to higher delays.

In [0]:
%sql
SELECT 
    ORIGIN_AIRPORT,
    COUNT(*) AS total_flights,
    ROUND(SUM(CASE WHEN ARRIVAL_DELAY > 0 THEN 1 ELSE 0 END) / COUNT(*),4) AS delay_rate,
    ROUND(AVG(CANCELLED),4) AS cancel_rate,
    ROUND(AVG(ORIGIN_CONGESTION),2) AS avg_congestion
FROM tables.default.flights_cleaned
GROUP BY ORIGIN_AIRPORT
HAVING COUNT(*) > 500
ORDER BY delay_rate DESC
LIMIT 20;


ORIGIN_AIRPORT,total_flights,delay_rate,cancel_rate,avg_congestion
RHI,954,0.4822,0.0157,296.82
COD,665,0.4707,0.018,220.33
OME,663,0.4676,0.0407,249.97
DLH,1719,0.4526,0.0303,225.58
OTZ,663,0.4465,0.0332,189.74
DAL,59699,0.4458,0.0142,3772.11
CIU,613,0.4388,0.0261,226.15
OAK,42315,0.435,0.0135,2709.45
CRW,2385,0.4331,0.0247,400.86
BTR,7164,0.429,0.0228,545.19


### Observations

- Major hub airports exhibit higher congestion metrics.
- Elevated congestion correlates with increased delay rates.
- Infrastructure and traffic density likely influence performance.

## 3. Monthly Delay and Cancellation Trends

This section evaluates seasonality in operational performance.

In [0]:
%sql
SELECT 
    MONTH,
    ROUND(SUM(CASE WHEN ARRIVAL_DELAY > 0 THEN 1 ELSE 0 END)/COUNT(*),4) AS delay_rate,
    ROUND(AVG(ARRIVAL_DELAY),2) AS avg_arrival_delay,
    ROUND(AVG(CANCELLED),4) AS cancel_rate
    
FROM tables.default.flights_cleaned
GROUP BY MONTH
ORDER BY MONTH;



MONTH,delay_rate,avg_arrival_delay,cancel_rate
1,0.3896,5.59,0.0255
2,0.4087,7.81,0.0478
3,0.377,4.74,0.0218
4,0.3541,3.09,0.0093
5,0.3524,4.37,0.0115
6,0.4107,9.33,0.0181
7,0.3835,6.33,0.0092
8,0.3543,4.5,0.0099
9,0.287,-0.8,0.0045
10,0.2905,-0.82,0.005


### Observations

- Seasonal variation is observed in delay frequency.
- Winter months may show higher cancellation rates.
- Travel demand peaks influence operational pressure.

In [0]:
%sql
SELECT 
    DEPARTURE_HOUR,
    ROUND(SUM(CASE WHEN ARRIVAL_DELAY > 0 THEN 1 ELSE 0 END) / COUNT(*),4) AS delay_rate
FROM tables.default.flights_cleaned
GROUP BY DEPARTURE_HOUR
ORDER BY DEPARTURE_HOUR;


DEPARTURE_HOUR,delay_rate
0,0.3312
1,0.3726
2,0.343
3,0.3085
4,0.3465
5,0.2056
6,0.2419
7,0.2733
8,0.2934
9,0.3154


### Observations

- Delay rates increase progressively across the day.
- Peak disruption observed in late afternoon and evening.
- Accumulated delays likely propagate into later departures.

## 5. Route-Level Bottleneck Analysis

This section identifies routes with persistent delay risk.

In [0]:
%sql
SELECT 
    CONCAT(ORIGIN_AIRPORT,'-',DESTINATION_AIRPORT) AS ROUTE,
    COUNT(*) AS total_flights,
    ROUND(SUM(CASE WHEN ARRIVAL_DELAY > 0 THEN 1 ELSE 0 END) / COUNT(*),4) AS delay_rate,
    ROUND(AVG(CANCELLED),4) AS cancel_rate
FROM tables.default.flights_cleaned
GROUP BY CONCAT(ORIGIN_AIRPORT,'-',DESTINATION_AIRPORT)
HAVING COUNT(*) > 300
ORDER BY delay_rate DESC
LIMIT 20;


ROUTE,total_flights,delay_rate,cancel_rate
DFW-OGG,504,0.6468,0.0119
ORD-HNL,334,0.6347,0.015
HOU-JAX,373,0.6247,0.008
ORD-BOI,543,0.6243,0.0295
DFW-HNL,695,0.623,0.0029
DEN-HNL,334,0.6138,0.006
IAH-HNL,334,0.6048,0.006
LAS-LBB,334,0.6018,0.009
LAS-FLL,1033,0.5973,0.0058
ATL-TTN,325,0.5969,0.0123


# Overall Descriptive Insights

- Airline-level performance varies significantly across carriers.
- Hub congestion contributes to delay propagation.
- Seasonal and hourly trends reveal structural operational pressure.
- Certain routes consistently demonstrate higher delay risk.

These descriptive findings establish the foundation for statistical validation and predictive modeling.